# IOC Miner — Benchmark Evaluation

Evaluates ioc-miner extractors (regex, SecBERT NER, baselines) against the CyNER
ground truth dataset and produces P/R/F1 tables for the paper.

**Runtime:** CPU is fine — no GPU needed for this notebook.

**Sections:**
1. Setup — install deps, clone repo, mount Drive
2. Prepare eval dataset — CyNER → benchmark JSONL
3. Baseline comparison — regex vs iocextract vs ioc-finder
4. SecBERT NER benchmark — load fine-tuned model from Drive (optional)
5. Verdict accuracy — ContextClassifier evaluation
6. PDN case study — run pipeline on uploaded report files
7. Save results to Drive

## 1. Setup

In [ ]:
# Clone repo (or pull latest if already exists)
import os
if os.path.exists("/content/ioc-miner"):
    !git -C /content/ioc-miner pull
else:
    !git clone https://github.com/calvinkatoroy/ioc-miner.git /content/ioc-miner
%cd /content/ioc-miner


In [ ]:
%%capture
# Core dependencies (no torch needed for regex-only benchmark)
!pip install datasets transformers tokenizers
!pip install pdfplumber beautifulsoup4 lxml requests
!pip install nltk tldextract validators regex
!pip install rich typer

# Baseline tools
!pip install iocextract
!pip install ioc-finder

# Install ioc-miner itself in editable mode
!pip install -e . --quiet

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)
print('Setup complete.')

In [ ]:
# Mount Google Drive to persist results across sessions
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ioc-miner'
os.makedirs(f'{DRIVE_DIR}/eval',    exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print('Drive mounted. Results will be saved to:', DRIVE_DIR)

## 2. Prepare Evaluation Dataset (CyNER → JSONL)

Downloads CyNER from HuggingFace and converts span annotations to ioc-miner's
benchmark JSONL format. Takes ~2 minutes on first run (downloads ~10 MB).

Output: `data/eval/cyner_eval.jsonl` — one sentence per line with IOC annotations.

In [ ]:
!mkdir -p data/eval

!python scripts/prepare_eval_data.py \
    --source cyner \
    --split train \
    --output data/eval/cyner_eval.jsonl \
    --stats

In [ ]:
# Inspect a few samples
import json

with open('data/eval/cyner_eval.jsonl') as f:
    samples = [json.loads(l) for l in f if l.strip()][:5]

for s in samples:
    if s['iocs']:  # only show samples with IOCs
        print(f"Text : {s['text'][:100]}")
        for ioc in s['iocs']:
            print(f"  IOC: {ioc['type']:<10} {ioc['value']}")
        print()

## 3. Baseline Comparison

Runs ioc-miner regex extractor against `iocextract` and `ioc-finder`.

Expected runtime: 3–8 minutes depending on dataset size.

In [ ]:
# Regex only — sanity check before baselines
!python scripts/run_benchmark.py \
    --eval data/eval/cyner_eval.jsonl \
    --extractors regex \
    --output results/regex_only.json

In [ ]:
# Full comparison: regex + iocextract + ioc-finder
# (cacador requires Go binary — skip for now)
!python scripts/run_benchmark.py \
    --eval data/eval/cyner_eval.jsonl \
    --extractors regex iocextract ioc-finder \
    --output results/baseline_comparison.json

In [ ]:
# Render results as a clean table for paper
import json
from ioc_miner.evaluation.benchmark import BenchmarkEvaluator, EvalResults, ExtractionMetrics
from ioc_miner.models.ioc import IOCType

with open('results/baseline_comparison.json') as f:
    raw = json.load(f)

# Reconstruct EvalResults objects
def _from_dict(d: dict) -> EvalResults:
    per_type = {}
    for k, v in d['per_type'].items():
        per_type[IOCType(k)] = ExtractionMetrics(tp=v['tp'], fp=v['fp'], fn=v['fn'])
    micro_d = d['micro']
    micro = ExtractionMetrics(tp=micro_d['tp'], fp=micro_d['fp'], fn=micro_d['fn'])
    return EvalResults(per_type=per_type, micro=micro)

comparison = {name: _from_dict(v) for name, v in raw['extraction'].items()}

print(BenchmarkEvaluator.comparison_markdown(comparison))

## 4. SecBERT NER Benchmark (Optional)

Requires a fine-tuned model saved to Drive from `colab_finetune.ipynb`.

Also needs `torch` and `transformers` with a GPU for reasonable speed — switch to
**Runtime → T4 GPU** before running this section.

In [ ]:
import os

MODEL_DIR = f'{DRIVE_DIR}/secbert-ioc-ner'

if os.path.exists(MODEL_DIR):
    print('Model found:', MODEL_DIR)
    print('Files:', os.listdir(MODEL_DIR))
else:
    print('Model NOT found at', MODEL_DIR)
    print('Run colab_finetune.ipynb first, then come back to this section.')

In [ ]:
# Run only if model exists
import os
if os.path.exists(MODEL_DIR):
    !python scripts/run_benchmark.py \
        --eval data/eval/cyner_eval.jsonl \
        --extractors regex secbert iocextract ioc-finder \
        --model {MODEL_DIR} \
        --output results/full_comparison.json
else:
    print('Skipping — no fine-tuned model found.')

## 5. Verdict Accuracy (ContextClassifier)

Evaluates the dual-stage context classifier (rule-based → NLI fallback).

**Note:** CyNER does not include verdict labels (malicious/benign), so the default
JSONL has `"verdict": "unknown"` for all entries — verdict evaluation will show
0 evaluated IOCs. To run this section meaningfully:

1. Take a subset of `cyner_eval.jsonl`
2. Manually annotate `verdict` fields (e.g. `"malicious"` for C2 IPs)
3. Save as `data/eval/verdict_labeled.jsonl`
4. Run the cell below against that file

Alternatively, the PDN case study (Section 6) provides a natural verdict ground truth
from the BSSN advisory.

In [ ]:
# Upload verdict_labeled.jsonl from your local machine
# (skip if already on Drive)
import os
from google.colab import files

os.makedirs("data/eval", exist_ok=True)
VERDICT_GT = "data/eval/verdict_labeled.jsonl"

if os.path.exists(VERDICT_GT):
    print("verdict_labeled.jsonl already exists — skipping upload.")
elif os.path.exists(f"{DRIVE_DIR}/eval/verdict_labeled.jsonl"):
    import shutil
    shutil.copy(f"{DRIVE_DIR}/eval/verdict_labeled.jsonl", VERDICT_GT)
    print("Copied from Drive.")
else:
    print("Upload verdict_labeled.jsonl:")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        with open(VERDICT_GT, "wb") as f:
            f.write(content)
        print(f"Saved to {VERDICT_GT}")


In [ ]:
VERDICT_GT = 'data/eval/verdict_labeled.jsonl'

import os
if os.path.exists(VERDICT_GT):
    !python scripts/run_benchmark.py \
        --eval {VERDICT_GT} \
        --extractors regex \
        --verdict \
        --output results/verdict_eval.json
else:
    print(f'{VERDICT_GT} not found.')
    print('Create a verdict-labeled JSONL to run this section.')
    print()
    print('Example format:')
    print('{"text": "C2 beacon to 185.220.101.47", "iocs": [{"type": "ip", "value": "185.220.101.47", "verdict": "malicious"}]}')

In [ ]:
# Same as above but with NLI zero-shot classifier enabled
# Slower (~2s per IOC on CPU) but resolves more UNKNOWN cases
import os
if os.path.exists(VERDICT_GT):
    !python scripts/run_benchmark.py \
        --eval {VERDICT_GT} \
        --extractors regex \
        --verdict --use-ml \
        --output results/verdict_eval_nli.json
else:
    print('Skipping — verdict_labeled.jsonl not found.')

## 6. PDN Case Study

Runs the full ioc-miner pipeline on PDN ransomware report files.

**Before running this section:**
1. Download the following public reports:
   - BSSN Brain Cipher advisory (PDF) — search `"ADVISORY-BRAIN-CIPHER BSSN 2024"`
   - SentinelOne / Recorded Future / Trend Micro blog posts (save as PDF or HTML)
2. Upload them to Colab using the cell below, or copy from Drive

The script deduplicates IOCs across all files and outputs a CSV + summary table.

In [ ]:
# Option A: Upload files from your local machine
from google.colab import files
import os

os.makedirs('data/case_study/pdn', exist_ok=True)
print('Select your PDN report files to upload (.pdf / .html / .txt):')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'data/case_study/pdn/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'Saved: {dest}')

In [ ]:
# Option B: Copy from Drive (if already uploaded there)
import shutil, os

DRIVE_PDN = f'{DRIVE_DIR}/case_study/pdn'
os.makedirs('data/case_study/pdn', exist_ok=True)

if os.path.exists(DRIVE_PDN):
    for fname in os.listdir(DRIVE_PDN):
        shutil.copy(f'{DRIVE_PDN}/{fname}', f'data/case_study/pdn/{fname}')
        print(f'Copied: {fname}')
else:
    print(f'Drive path not found: {DRIVE_PDN}')
    print('Upload files there first, or use Option A above.')

In [ ]:
import os
pdn_files = [f for f in os.listdir('data/case_study/pdn') if not f.startswith('.')]

if not pdn_files:
    print('No files in data/case_study/pdn/ — upload reports first.')
else:
    print(f'Running on {len(pdn_files)} files:', pdn_files)
    !python scripts/case_study_pdn.py \
        --input data/case_study/pdn/ \
        --output results/pdn_iocs.csv

In [ ]:
# Preview extracted IOCs
import os
if os.path.exists('results/pdn_iocs.csv'):
    with open('results/pdn_iocs.csv') as f:
        lines = f.readlines()

    print(f'Total: {len(lines)-1} IOCs')
    print()
    print(''.join(lines[:20]))  # header + first 19
else:
    print('No output yet — run the case study cell above first.')

## 7. Save Results to Drive

In [ ]:
import shutil, os, glob

result_files = glob.glob('results/*.json') + glob.glob('results/*.csv')

for fpath in result_files:
    dest = f'{DRIVE_DIR}/results/{os.path.basename(fpath)}'
    shutil.copy(fpath, dest)
    print(f'Saved: {dest}')

# Also save the eval JSONL to Drive for future sessions
if os.path.exists('data/eval/cyner_eval.jsonl'):
    shutil.copy('data/eval/cyner_eval.jsonl', f'{DRIVE_DIR}/eval/cyner_eval.jsonl')
    print(f'Saved: {DRIVE_DIR}/eval/cyner_eval.jsonl')

print('\nAll results saved to Drive.')

In [ ]:
# Download results directly to your local machine
from google.colab import files
import glob

for fpath in glob.glob('results/*.json') + glob.glob('results/*.csv'):
    files.download(fpath)